In [2]:
import numpy as np
import scipy.stats as stats

In [3]:
# Set random seed for identical results
np.random.seed(42)

# Generate latency data for 4 different KV-caching setups
# Let's say Setup D is secretly the fastest one (lower latency mean)
group_A = np.random.normal(loc=50, scale=8, size=100) # Baseline
group_B = np.random.normal(loc=50, scale=8, size=100) # Variant 1 (No real change)
group_C = np.random.normal(loc=49.5, scale=8, size=100) # Variant 2 (Tiny change)
group_D = np.random.normal(loc=45, scale=8, size=100) # Variant 3 (Genuinely faster!)

In [11]:
f_stat, p_stat = stats.f_oneway(group_A, group_B, group_C, group_D) # One way Anova Test

In [12]:
print(f"F stat : {f_stat:.2f}")
print(f"p_stat : {p_stat*100:.4f}")

F stat : 6.88
p_stat : 0.0158


It protects your overall error rate by saying: "Hey! Stop coding, look over here! There is a genuine difference somewhere in this pile of 4 groups!" But notice what it doesn't tell you: It doesn't tell you which group is the winner. Is Group D beating Group A? Is Group C actually worse than Group B? ANOVA leaves you completely blind to the exact pairing.

# PairWise Turkey Test

In [15]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [17]:
all_groups = np.concatenate([group_A, group_B, group_C, group_D])

In [19]:
group_labels = (
    ["Group A"] * 100 +
    ["Group B"] * 100 +
    ["Group C"] * 100 +
    ["Group D"] * 100
) 

In [20]:
turkey_result = pairwise_tukeyhsd(endog=all_groups, groups=group_labels, alpha=0.05)

In [21]:
print(turkey_result)

 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 group1  group2 meandiff p-adj   lower   upper  reject
------------------------------------------------------
Group A Group B   1.0092 0.7896 -1.7949  3.8133  False
Group A Group C   0.8499 0.8626 -1.9542  3.6541  False
Group A Group D  -3.3145  0.013 -6.1186 -0.5104   True
Group B Group C  -0.1593 0.9989 -2.9634  2.6448  False
Group B Group D  -4.3237 0.0005 -7.1278 -1.5196   True
Group C Group D  -4.1644 0.0008 -6.9686 -1.3603   True
------------------------------------------------------


In [23]:
# meandiff = group 2 - group 1

# Statistical Analysis Report: Latency Optimization

## 1. Core Column Definitions

| Column | Definition | Interpretation |
| :--- | :--- | :--- |
| **`meandiff`** | Absolute difference between group means (`Group2 - Group1`). Measured in milliseconds. | **Negative:** Group 2 is faster.<br>**Positive:** Group 2 is slower. |
| **`p-adj`** | Adjusted p-value from Statsmodels. | Penalizes for multiple comparisons (6 tests). Compare against baseline $\alpha = 0.05$. |
| **`lower` / `upper`** | 95% Confidence Interval (CI) of the true difference. | If the range includes **0**, the difference is not statistically significant. |
| **`reject`** | Final decision flag. | **True:** Real, solid performance difference.<br>**False:** Difference is likely random noise. |

---

## 2. Engineering Blueprint: Row Analysis

### ❌ The Dead Zones (Group A, B, and C)
*Comparisons: A vs B, A vs C, and B vs C*

*   **Status:** `reject = False` for all pairings.
*   **Significance:** Adjusted p-values are extremely high (e.g., 0.7896, 0.8626, 0.9989).
*   **Confidence Intervals:** All intervals straddle zero (e.g., A vs B ranges from `-1.79` to `+3.81`).
    *   *Implication:* The math indicates a high probability that the true difference is **0**.
*   **Verdict:** Variants **B** and **C** are ineffective. **Do not ship.**

### 🚀 The Clear Winner (Group D)
*Comparisons: A vs D, B vs D, and C vs D*

Every test involving Group D shows a statistically significant improvement:

| Comparison | Mean Diff | p-adj | Reject | CI Range |
| :--- | :--- | :--- | :--- | :--- |
| **A vs D** | `-3.31` | `0.013` | **True** | `-6.11` to `-0.51` |
| **B vs D** | `-4.32` | `0.0005` | **True** | Excludes 0 |
| **C vs D** | `-4.16` | `0.0008` | **True** | Excludes 0 |

*   **Consistency:** All `reject` flags are **True**.
*   **Direction:** Significantly negative `meandiff` values prove Group D reduces latency compared to the baseline (Group A) and all other variants.
*   **Confidence:** The 95% CI for A vs D (`-6.11` to `-0.51`) completely excludes zero.
    *   *Implication:* We are 95% confident Group D reduces latency by **at least 0.51 ms** and up to **6.11 ms**.

---

## 3. Final Verdict

**Action:** Merge **Group D** to the master branch and deploy to production immediately.

By utilizing ANOVA followed by Tukey's HSD, you have:
1.  Filtered out the noise of Variants B and C.
2.  Protected against false-positive inflation.
3.  Isolated the true optimization winner with total statistical certainty.